In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


PyTorch version: 2.7.1+cu126
CUDA available: True
torch.version.cuda: 12.6
GPU: NVIDIA GeForce RTX 4090 Laptop GPU


In [6]:
# Minimal JSON (array) -> Parquet using pyarrow only

import json
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

SRC = "newsapi_ai_all_topics_20250806.json"
DST = "test.parquet"

def parse_iso(ts: str):
    if not ts:
        return None
    try:
        # handles ...Z by replacing with UTC offset
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))
    except Exception:
        return None

# 1) Load the JSON array from disk
with open(SRC, "r", encoding="utf-8") as f:
    records = json.load(f)  # expects a top-level list[dict]

# 2) Normalize to just the 6 columns the KG stage expects
rows = []
for r in records:
    rows.append({
        "doc_id": str(r.get("doc_id", "")),
        "title": r.get("title", "") or "",
        "content": r.get("content", "") or "",
        "source": r.get("source", "") or "",
        "published_at": parse_iso(r.get("published_at") or r.get("publishedAt") or ""),
        "url": r.get("url") or r.get("link") or "",
    })

# 3) Define the Arrow schema (timestamp in milliseconds)
schema = pa.schema([
    ("doc_id", pa.string()),
    ("title", pa.string()),
    ("content", pa.string()),
    ("source", pa.string()),
    ("published_at", pa.timestamp("ms")),
    ("url", pa.string()),
])

# 4) Build the table and write Parquet (Snappy)
table = pa.Table.from_pylist(rows, schema=schema)
pq.write_table(table, DST, compression="snappy")

print(f"✅ Wrote {table.num_rows:,} rows to {DST}")
print("Columns:", [f.name for f in table.schema])


✅ Wrote 100 rows to test.parquet
Columns: ['doc_id', 'title', 'content', 'source', 'published_at', 'url']


In [4]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_REPO = "Qwen/Qwen1.7b-1.5B-Instruct-AWQ"  # <-- change if needed
MODEL_DIR = Path("./models/Qwen2.5-1.5B-Instruct-AWQ")

# MODEL_DIR.mkdir(parents=True, exist_ok=True)
# snapshot_download(
#     repo_id=MODEL_REPO,
#     local_dir=str(MODEL_DIR),
#     local_dir_use_symlinks=False,
#     ignore_patterns=["*.bin", "*.gguf"]  # prefer safetensors
# )
print("Model ready at:", MODEL_DIR.resolve())


Model ready at: /home/jroberts/kg_extract/kge/models/Qwen2.5-1.5B-Instruct-AWQ


In [5]:
import hashlib, re, json
from typing import Dict

def ent_id_from_name_type(name: str, etype: str) -> str:
    key = f"{etype}|{name.strip().lower()}"
    return "ent:" + hashlib.sha1(key.encode()).hexdigest()

def chunk_text(s: str, max_chars: int = 2500):
    # simple char-based chunker for dev; good enough for 1.5B
    s = s.strip()
    return [s[i:i+max_chars] for i in range(0, len(s), max_chars)] or [""]

# Strict JSON schema for guided decoding
JSON_SCHEMA: Dict = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {
                        "type": "string",
                        "enum": ["Company","Person","Technology","Agency","Program","Contract","System","Location"]
                    },
                    "description": {"type": "string"}
                },
                "required": ["name","type"]
            }
        },
        "relationships": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "source": {"type": "string"},
                    "relation": {
                        "type": "string",
                        "enum": ["PARTNERS_WITH","EMPLOYS","DEVELOPS","FUNDS","COMPETES_WITH","SUPPLIES","USES","LOCATED_IN"]
                    },
                    "target": {"type": "string"},
                    "confidence": {"type":"number","minimum":0,"maximum":1}
                },
                "required": ["source","relation","target"]
            }
        }
    },
    "required": ["entities","relationships"]
}


In [6]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
import os
os.environ["VLLM_DISABLE_MAMBA"] = "1"          # skip compiling Mamba kernels
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"  # Less than Python 3.14
os.environ["VLLM_DISABLE_TORCH_COMPILE"] = "1"   # avoid TorchInductor edge cases

# JSON_SCHEMA = {
#     "type": "object",
#     "properties": {
#         "entities": {"type": "array", "items": {"type": "object",
#             "properties": {"name": {"type": "string"},
#                            "type": {"type": "string"},
#                            "description": {"type": "string"}},
#             "required": ["name", "type"]}},
#         "relationships": {"type": "array", "items": {"type": "object",
#             "properties": {"source": {"type": "string"},
#                            "relation": {"type": "string"},
#                            "target": {"type": "string"},
#                            "confidence": {"type": "number"}},
#             "required": ["source", "relation", "target"]}}
#     },
#     "required": ["entities", "relationships"]
# }


llm = LLM(
    model=str(MODEL_DIR),
    quantization="awq_marlin",
    tensor_parallel_size=1,         # single RTX 4090
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    trust_remote_code=True,         # Qwen repos often need this
    download_dir=str(MODEL_DIR),    # keeps everything local
    guided_decoding_backend="xgrammar"
)

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=384,
    guided_decoding=GuidedDecodingParams(
        json=JSON_SCHEMA,
    ),
)

# test
out = llm.generate(["Return a JSON object with a single key 'ping' set to 'pong'."], sampling)[0]
print(out.outputs[0].text)

INFO 08-18 09:41:07 [config.py:1604] Using max model len 8192
INFO 08-18 09:41:07 [awq_marlin.py:116] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 08-18 09:41:09 [config.py:2434] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 08-18 09:41:09 [__init__.py:2899] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 08-18 09:41:12 [__init__.py:235] Automatically detected platform cuda.
INFO 08-18 09:41:14 [core.py:572] Waiting for init message from front-end.
INFO 08-18 09:41:14 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='models/Qwen2.5-1.5B-Instruct-AWQ', speculative_config=None, tokenizer='models/Qwen2.5-1.5B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.22s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.22s/it]



INFO 08-18 09:41:17 [default_loader.py:262] Loading weights took 1.28 seconds
INFO 08-18 09:41:18 [gpu_model_runner.py:1892] Model loading took 1.1018 GiB and 1.669098 seconds
INFO 08-18 09:41:24 [backends.py:530] Using cache directory: /home/jroberts/.cache/vllm/torch_compile_cache/88a5c3a786/rank_0_0/backbone for vLLM's torch.compile
INFO 08-18 09:41:24 [backends.py:541] Dynamo bytecode transform time: 6.02 s
INFO 08-18 09:41:28 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 3.656 s
INFO 08-18 09:41:29 [monitor.py:34] torch.compile takes 6.02 s in total
INFO 08-18 09:41:30 [gpu_worker.py:255] Available KV cache memory: 11.07 GiB
INFO 08-18 09:41:30 [kv_cache_utils.py:833] GPU KV cache size: 414,384 tokens
INFO 08-18 09:41:30 [kv_cache_utils.py:837] Maximum concurrency for 8,192 tokens per request: 50.58x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:02<00:00, 24.70it/s]


INFO 08-18 09:41:33 [gpu_model_runner.py:2485] Graph capturing finished in 3 secs, took 0.56 GiB
INFO 08-18 09:41:33 [core.py:193] init engine (profile, create kv cache, warmup model) took 15.57 seconds


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

{
  "entities": [
    {
      "name": "John",
      "type": "Person"
    },
    {
      "name": "Jane",
      "type": "Person"
    }
  ]
  , "relationships": [
    {
      "source": "John",
      "relation": "LOCATED_IN",
      "target": "Jane"
    }
  ]
  }


In [1]:
# Option B: load a local Parquet table (uncomment & adjust path)
import duckdb
parquet_path = "./test.parquet"
with duckdb.connect() as con:
    df = con.execute(f"""
        SELECT doc_id, title, content, source, published_at, url
        FROM '{parquet_path}'
        WHERE content IS NOT NULL AND length(content) > 100
        LIMIT 50
    """).fetch_df()


In [85]:
import os, socket, time, subprocess
from pathlib import Path

# vLLM perf/env flags
os.environ["VLLM_DISABLE_MAMBA"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"      # OK on Python < 3.14
os.environ["VLLM_DISABLE_TORCH_COMPILE"] = "1"

# Single checkpoint (GPTQ-Int8)
MODEL_REPO = "Qwen/Qwen3-1.7B-GPTQ-Int8"
MODEL_DIR = Path("./models/qwen3-1_7b-gptq-int8"); MODEL_DIR.mkdir(parents=True, exist_ok=True)

# vLLM OpenAI-compatible server settings
VLLM_PORT = int(os.environ.get("VLLM_PORT", "8000"))
BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
SERVED_MODEL_NAME = "qwen3-1_7b-gptq"

# Reasoning parser for Qwen3 (exposes thinking content); you can keep server parser on
ENABLE_REASONING_OUTPUT = True  # you will still toggle thinking per-request

# Local parquet path w/ ~100 articles; must include: doc_id, title, content, source, published_at, url
PARQUET_PATH = "./test.parquet"
assert Path(PARQUET_PATH).exists(), f"Missing file: {PARQUET_PATH}"

print("MODEL_REPO:", MODEL_REPO)
print("MODEL_DIR :", MODEL_DIR.resolve())
print("BASE_URL  :", BASE_URL)
print("PARQUET   :", PARQUET_PATH)


MODEL_REPO: Qwen/Qwen3-1.7B-GPTQ-Int8
MODEL_DIR : /home/jroberts/kg_extract/kge/models/qwen3-1_7b-gptq-int8
BASE_URL  : http://127.0.0.1:8000/v1
PARQUET   : ./test.parquet


In [10]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,
    ignore_patterns=["*.bin", "*.gguf", "*.onnx"]
)
print("Model ready at:", MODEL_DIR.resolve())


/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Model ready at: /home/jroberts/kg_extract/kge/models/qwen3-1_7b-gptq-int8


In [135]:
# ---- HARD RESET any existing vLLM server on our port, then robust start ----
import os, sys, time, socket, subprocess, urllib.request, json, signal

VLLM_PORT = int(os.environ.get("VLLM_PORT", "8000"))
BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"

# 1) Kill any stray python process still bound to port (common after nvidia-smi kill)
def _pids_on_port(port: int):
    try:
        import psutil
    except Exception:
        # lightweight fallback using lsof (WSL/Linux)
        out = subprocess.run(["bash","-lc", f"lsof -t -i:{port}"], capture_output=True, text=True)
        return [int(x) for x in out.stdout.strip().split() if x.strip().isdigit()]
    else:
        pids = []
        for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
            try:
                for con in p.connections(kind="inet"):
                    if con.laddr.port == port:
                        pids.append(p.pid); break
            except Exception:
                pass
        return pids

def _kill_port_users(port: int):
    pids = _pids_on_port(port)
    if not pids:
        return
    print(f"Found {len(pids)} process(es) on :{port} -> killing...")
    for pid in pids:
        try: os.kill(pid, signal.SIGTERM)
        except Exception: pass
    # wait a moment, then SIGKILL if still there
    time.sleep(2)
    for pid in _pids_on_port(port):
        try: os.kill(pid, signal.SIGKILL)
        except Exception: pass
    time.sleep(1)

_kill_port_users(VLLM_PORT)

# 2) Start a fresh server
FAST_START = False   # True adds --enforce-eager (faster dev startup; slightly lower peak throughput)


serve_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--model", str(MODEL_DIR),
    "--served-model-name", SERVED_MODEL_NAME,
    "--enable-prefix-caching",
    "--enable-chunked-prefill", 
    # "--attention-backend", "flashinfer",
    "--max-model-len", "8192",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.85",
    "--quantization", "gptq_marlin",
    "--guided-decoding-backend", "xgrammar",
    # "--reasoning-parser", "deepseek_r1",  # Qwen3 thinking parser
]
if FAST_START:
    serve_cmd += ["--enforce-eager"]  # recommended for debugging / faster first boot

# During dev, avoid stale compile cache confusion
os.environ.setdefault("VLLM_DISABLE_COMPILE_CACHE", "1")

print("Launching:", " ".join(serve_cmd))
vllm_proc = subprocess.Popen(serve_cmd)

# 3) Wait until TCP port is listening
def _port_open(port: int) -> bool:
    s = socket.socket(); s.settimeout(0.2)
    try:
        s.connect(("127.0.0.1", port)); return True
    except Exception:
        return False
    finally:
        s.close()

start = time.time()
while time.time() - start < 90:
    if _port_open(VLLM_PORT):
        break
    time.sleep(1)
else:
    raise RuntimeError("vLLM port never opened")

# 4) Wait for /health (API ready)
def _http_ok(url: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=3) as r: return r.status == 200
    except Exception: return False

start = time.time()
while time.time() - start < 300:  # allow up to 5 minutes for first boot
    if _http_ok(f"http://127.0.0.1:{VLLM_PORT}/health"):
        print("vLLM /health OK")
        break
    time.sleep(1)
else:
    raise RuntimeError("vLLM never reported healthy")

# 5) Extra sanity: ensure the served model is visible via /v1/models
def _list_models():
    with urllib.request.urlopen(f"{BASE_URL}/models", timeout=5) as r:
        return json.loads(r.read())

models = _list_models()
ids = [m.get("id") for m in models.get("data", [])]
print("Models:", ids)

assert SERVED_MODEL_NAME in ids, f"{SERVED_MODEL_NAME} not in /v1/models -> got {ids}"
print("vLLM server is up and model is listed.")


Launching: python -m vllm.entrypoints.openai.api_server --host 127.0.0.1 --port 8000 --model models/qwen3-1_7b-gptq-int8 --served-model-name qwen3-1_7b-gptq --enable-prefix-caching --enable-chunked-prefill --max-model-len 8192 --trust-remote-code --gpu-memory-utilization 0.85 --quantization gptq_marlin --guided-decoding-backend xgrammar
INFO 08-19 13:06:39 [__init__.py:235] Automatically detected platform cuda.
INFO 08-19 13:06:41 [api_server.py:1755] vLLM API server version 0.10.0
INFO 08-19 13:06:41 [cli_args.py:261] non-default args: {'host': '127.0.0.1', 'model': 'models/qwen3-1_7b-gptq-int8', 'trust_remote_code': True, 'max_model_len': 8192, 'quantization': 'gptq_marlin', 'served_model_name': ['qwen3-1_7b-gptq'], 'guided_decoding_backend': 'xgrammar', 'gpu_memory_utilization': 0.85, 'enable_prefix_caching': True, 'enable_chunked_prefill': True}
INFO 08-19 13:06:47 [config.py:1604] Using max model len 8192
INFO 08-19 13:06:48 [gptq_marlin.py:170] The model is convertible to gptq_ma

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.96s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.96s/it]



INFO 08-19 13:06:54 [default_loader.py:262] Loading weights took 1.99 seconds
INFO 08-19 13:06:55 [gpu_model_runner.py:1892] Model loading took 1.9232 GiB and 2.362724 seconds
INFO 08-19 13:07:04 [backends.py:528] vLLM's torch.compile cache is disabled.
INFO 08-19 13:07:04 [backends.py:541] Dynamo bytecode transform time: 8.62 s
INFO 08-19 13:07:25 [backends.py:215] Compiling a graph for dynamic shape takes 20.33 s
INFO 08-19 13:07:26 [monitor.py:34] torch.compile takes 28.95 s in total
INFO 08-19 13:07:27 [gpu_worker.py:255] Available KV cache memory: 10.26 GiB
INFO 08-19 13:07:28 [kv_cache_utils.py:833] GPU KV cache size: 96,032 tokens
INFO 08-19 13:07:28 [kv_cache_utils.py:837] Maximum concurrency for 8,192 tokens per request: 11.72x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:03<00:00, 21.27it/s]


INFO 08-19 13:07:31 [gpu_model_runner.py:2485] Graph capturing finished in 4 secs, took 0.69 GiB
INFO 08-19 13:07:31 [core.py:193] init engine (profile, create kv cache, warmup model) took 36.58 seconds
INFO 08-19 13:07:32 [loggers.py:141] Engine 000: vllm cache_config_info with initialization after num_gpu_blocks is: 6002
WARNING 08-19 13:07:32 [config.py:1528] Default sampling parameters have been overridden by the model's Hugging Face generation config recommended from the model creator. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
INFO 08-19 13:07:32 [serving_responses.py:89] Using default chat sampling params from model: {'temperature': 0.6, 'top_k': 20, 'top_p': 0.95}
INFO 08-19 13:07:32 [serving_chat.py:122] Using default chat sampling params from model: {'temperature': 0.6, 'top_k': 20, 'top_p': 0.95}
INFO 08-19 13:07:32 [serving_completion.py:77] Using default completion sampling params from model: {'temperature': 0.6, 'top_k': 20, 't

INFO:     Started server process [141914]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


INFO:     127.0.0.1:40202 - "GET /health HTTP/1.1" 200 OK
vLLM /health OK
INFO:     127.0.0.1:40216 - "GET /v1/models HTTP/1.1" 200 OK
Models: ['qwen3-1_7b-gptq']
vLLM server is up and model is listed.


In [136]:
# One-token sanity request with a tight timeout: if this returns, the engine is genuinely serving.
from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key="local-anything")

resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[{"role":"user","content":"Return the word PONG."}],
    temperature=0.0,
    max_tokens=8,
    # Non-thinking to minimize overhead on first call:
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
    timeout=20,   # fail fast if engine is wedged
)
print("Ping:", resp.choices[0].message.content)


INFO 08-19 13:07:39 [chat_utils.py:473] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 08-19 13:07:39 [logger.py:41] Received request chatcmpl-9c65cb670b6f4b3eaaa148ca8a4a2e36: prompt: '<|im_start|>user\nReturn the word PONG.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=0, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=8, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None), prompt_token_ids: None, prompt_embeds shape: None, lora_request: None.
INFO 08-19 13:07:39 [async_llm.py:269] Added request chatcmpl-9c65cb670b6f4b3eaaa148ca8a4a2e36.
INFO:     127.0.0.1:40

INFO 08-19 13:07:42 [loggers.py:122] Engine 000: Avg prompt throughput: 1.7 tokens/s, Avg generation throughput: 0.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 08-19 13:07:52 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [44]:
from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key="local-anything")

# Non-thinking
r1 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[{"role":"user","content":"Summarize: Large language models enable KG extraction from text in three bullets."}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},  # toggle off
    temperature=0.0,
)
print("non-thinking: \n", r1.choices[0].message.content)

# Thinking
r2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[{"role":"user","content":"Summarize: Large language models enable KG extraction from text in three bullets."}],
    extra_body={"chat_template_kwargs":{"enable_thinking": True}},   # toggle on
    temperature=0.0,
    timeout=120
)
print("thinking:     ", r2.choices[0].message.content)
# Optional: reasoning stream (when parser is enabled)
# print("reasoning:", getattr(r2.choices[0].message, "reasoning_content", None))
print("finish_reason:", r2.choices[0].finish_reason)


INFO 08-18 16:58:06 [logger.py:41] Received request chatcmpl-d4e51dff96f748338a343cb6e28e666c: prompt: '<|im_start|>user\nSummarize: Large language models enable KG extraction from text in three bullets.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=0, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=8164, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None), prompt_token_ids: None, prompt_embeds shape: None, lora_request: None.
INFO 08-18 16:58:06 [async_llm.py:269] Added request chatcmpl-d4e51dff96f748338a343cb6e28e666c.
INFO:     127.0.0.1:54190 - "POST /v1/chat/completions HTTP/1.1" 200 OK
non-thinking: 
 - **Automatically extract struct

INFO 08-18 16:58:16 [loggers.py:122] Engine 000: Avg prompt throughput: 5.2 tokens/s, Avg generation throughput: 50.4 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 22.9%


In [45]:
import time
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key="local-anything")

def measure_once(prompt: str, *, thinking: bool, max_tokens: int = 512, timeout: int = 180):
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[{"role":"user","content": prompt}],
        extra_body={"chat_template_kwargs": {"enable_thinking": bool(thinking)}},  # per-request toggle
        temperature=0.0,
        max_tokens=max_tokens,
        timeout=timeout,
    )
    t1 = time.perf_counter()

    choice = resp.choices[0]
    out = {
        "final": choice.message.content or "",
        "reasoning": getattr(choice.message, "reasoning_content", None),
        "finish_reason": choice.finish_reason,
        "latency_sec": t1 - t0,  # total end-to-end time
        "usage": {
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "total_tokens": resp.usage.total_tokens,
        },
    }
    return out

# Example: measure both modes for the same prompt
prompt = "Summarize: Large language models enable KG extraction from text in three bullets."
non_thinking = measure_once(prompt, thinking=False)
thinking     = measure_once(prompt, thinking=True)

print("NON-THINKING:", non_thinking["latency_sec"], "s", non_thinking["usage"])
print("THINKING   :", thinking["latency_sec"], "s", thinking["usage"])
print("finish_reasons:", non_thinking["finish_reason"], "|", thinking["finish_reason"])


INFO 08-18 16:58:24 [logger.py:41] Received request chatcmpl-6797784064c146e1b2fac3a5cd9696af: prompt: '<|im_start|>user\nSummarize: Large language models enable KG extraction from text in three bullets.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=0, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=512, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None), prompt_token_ids: None, prompt_embeds shape: None, lora_request: None.
INFO 08-18 16:58:24 [async_llm.py:269] Added request chatcmpl-6797784064c146e1b2fac3a5cd9696af.
INFO:     127.0.0.1:35796 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 08-18 16:58:25 [logger.py:41] Received reque

In [96]:
# KGGen uses LiteLLM under the hood; we’ll point it at our local vLLM OpenAI endpoint.
# See KGGen README (model string format & base_url override).
# import importlib
# dspy = importlib.import_module("dspy")
# if not hasattr(dspy, "dspy"):
#     dspy.dspy = dspy  # satisfy bad type annotation used inside kg_gen<=0.1.x
    
from kg_gen import KGGen

# LiteLLM-compatible "OpenAI" provider; vLLM exposes OpenAI /v1 API at BASE_URL
KGGEN_MODEL = f"openai/{SERVED_MODEL_NAME}"

kg = KGGen(
    model=KGGEN_MODEL,
    temperature=0.0,
    api_key=os.environ.get("OPENAI_API_KEY", "EMPTY"),  # vLLM doesn't check by default
)

# KGGen allows base_url injection; some versions accept a constructor param, others read env.
# We'll set environment variables LiteLLM honors for OpenAI-compatible endpoints:
os.environ["OPENAI_API_BASE"] = BASE_URL           # many libs read this
os.environ["OPENAI_BASE_URL"] = BASE_URL           # fallback for some clients
os.environ["LITELLM_BASE_URL"] = BASE_URL          # LiteLLM override if needed
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "EMPTY")

print("KGGen ready against:", BASE_URL, " model:", KGGEN_MODEL)


KGGen ready against: http://127.0.0.1:8000/v1  model: openai/qwen3-1_7b-gptq


INFO 08-18 18:03:36 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [116]:
import duckdb
con = duckdb.connect()

tbl = con.execute(f"""
    SELECT doc_id, title, content, source, published_at, url
    FROM '{PARQUET_PATH}'
    WHERE content IS NOT NULL AND length(content) > 100
    LIMIT 100
""").fetch_arrow_table()

print(tbl.schema)
print("Rows:", tbl.num_rows)


doc_id: string
title: string
content: string
source: string
published_at: timestamp[us]
url: string
Rows: 100


INFO 08-19 12:27:04 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [ ]:
import pyarrow as pa

def rows_from_arrow(table: pa.Table):
    cols = [table.column(i) for i in range(table.num_columns)]
    names = [table.field(i).name for i in range(table.num_columns)]
    for r in range(table.num_rows):
        yield {names[c]: cols[c][r].as_py() for c in range(len(names))}

def compose_text(row: dict, use_soft_toggle: str = "") -> str:
    # Optional soft toggle: "/no_think " or "/think " (Qwen supports these chat-template cues)
    # If you prefer strict toggling, use extra_body (Cell 5) in your own client.
    title = (row.get("title") or "").strip()
    body  = (row.get("content") or "").strip()
    prefix = (use_soft_toggle.strip() + " ").strip()
    return f"{prefix}{title}\n\n{body}".strip()


INFO 08-18 17:46:12 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [78]:
import pyarrow as pa
import dspy



entity_rows, rel_rows = [], []

for row in rows_from_arrow(tbl):
    doc_id = row["doc_id"]
    text   = compose_text(row, use_soft_toggle="/no_think")  # force non-thinking for this pass

    CHUNK = 2500  # chars; tune 2500–5000
    # make sure this runs BEFORE any kg.generate() calls
    lm = dspy.LM(
        f"openai/{SERVED_MODEL_NAME}",
        api_base=BASE_URL,          # vLLM OpenAI-compatible server
        api_key="local-anything",
        max_tokens=1024,            # avoid huge generations that get truncated
        temperature=0.0,
        # When DSPy falls back to JSONAdapter, this hints JSON-only outputs
        response_format={"type": "json_object"},
    )
    dspy.settings.configure(lm=lm)

    # in your loop, change the generate() call:
    graph = kg.generate(
        input_data=text,
        context=("news article; "
                "ONLY return JSON outputs; "
                "cap to at most 80 relations"),  # keeps outputs bounded
        chunk_size=CHUNK,
        cluster=True,
        temperature=0.1,
    )

    ents = getattr(graph, "entities", None) or graph.get("entities", [])
    rels = getattr(graph, "relations", None) or graph.get("relations", [])

    for e in ents:
        if isinstance(e, dict):
            entity_rows.append({
                "doc_id": doc_id,
                "name": e.get("name") or e.get("text") or str(e),
                "type": e.get("type") or "",
                "description": e.get("description") or "",
            })
        else:
            entity_rows.append({"doc_id": doc_id, "name": str(e), "type": "", "description": ""})

    for trip in rels:
        if isinstance(trip, (list, tuple)) and len(trip) == 3:
            s, r, o = trip
            rel_rows.append({"doc_id": doc_id, "subject": str(s), "relation": str(r), "object": str(o)})
        elif isinstance(trip, dict):
            rel_rows.append({"doc_id": doc_id,
                             "subject": str(trip.get("subject", "")),
                             "relation": str(trip.get("relation", "")),
                             "object": str(trip.get("object", ""))})
print("non-thinking pass:", len(entity_rows), "entities,", len(rel_rows), "relations")


17:46:16 - LiteLLM:ERROR: litellm_logging.py:4483 - Error creating standard logging object - No module named 'fastapi_sso'
Traceback (most recent call last):
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 736, in completion
    raise e
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 664, in completion
    ) = self.make_sync_openai_chat_completion_request(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/logging_utils.py", line 149, in sync_wrapper
    result = func(*args, **kwargs)
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 482, in make_sync_openai_chat_completion_request
    raise e
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 464, in make_sync_openai_chat_completion_request
    raw

INFO 08-18 17:46:16 [logger.py:41] Received request chatcmpl-a8b7b92049bb416db004ae5490dbd5ce: prompt: '<|im_start|>system\nYour input fields are:\n1. `items` (set[str]): \n2. `context` (str): the larger context in which the items appear\nYour output fields are:\n1. `cluster` (set[str]):\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## items ## ]]\n{items}\n\n[[ ## context ## ]]\n{context}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "cluster": "{cluster}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"string\\"}, \\"uniqueItems\\": true}"\n}\nIn adhering to this structure, your objective is: \n        Find one cluster of related items from the list.\n        A cluster should contain items that are the same in meaning, with different tenses, plural forms, stem forms, or cases. \n        

2025/08/18 17:46:19 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/18 17:46:19 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


INFO 08-18 17:46:19 [logger.py:41] Received request chatcmpl-0a39440ec1aa40aa9bcbe66fab18166c: prompt: '<|im_start|>system\nYour input fields are:\n1. `items` (set[str]): \n2. `context` (str): the larger context in which the items appear\nYour output fields are:\n1. `cluster` (set[str]):\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## items ## ]]\n{items}\n\n[[ ## context ## ]]\n{context}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "cluster": "{cluster}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"string\\"}, \\"uniqueItems\\": true}"\n}\nIn adhering to this structure, your objective is: \n        Find one cluster of related items from the list.\n        A cluster should contain items that are the same in meaning, with different tenses, plural forms, stem forms, or cases. \n        

ValidationError: 1 validation error for set[str]
  Input should be a valid set [type=set_type, input_value={'engineers': 'engineers'...3D printing technology'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/set_type

In [63]:
import pyarrow as pa
import dspy

# If you want to compare, clear and rerun with thinking:
entity_rows_think, rel_rows_think = [], []

for row in rows_from_arrow(tbl):
    doc_id = row["doc_id"]
    text   = compose_text(row, use_soft_toggle="/think")  # soft-toggle thinking

    CHUNK = 2500  # chars; tune 2500–5000
    # make sure this runs BEFORE any kg.generate() calls
    lm = dspy.LM(
        f"openai/{SERVED_MODEL_NAME}",
        api_base=BASE_URL,          # vLLM OpenAI-compatible server
        api_key="local-anything",
        max_tokens=1024,            # avoid huge generations that get truncated
        temperature=0.0,
        # When DSPy falls back to JSONAdapter, this hints JSON-only outputs
        response_format={"type": "json_object"},
    )
    
    dspy.settings.configure(lm=lm)
    graph = kg.generate(
        input_data=text,
        context=("news article; "
                "ONLY return JSON outputs; "
                "cap to at most 80 relations"),  # keeps outputs bounded
        chunk_size=CHUNK,
        cluster=True,
        temperature=0.1,
    )

    ents = getattr(graph, "entities", None) or graph.get("entities", [])
    rels = getattr(graph, "relations", None) or graph.get("relations", [])

    for e in ents:
        if isinstance(e, dict):
            entity_rows_think.append({
                "doc_id": doc_id,
                "name": e.get("name") or e.get("text") or str(e),
                "type": e.get("type") or "",
                "description": e.get("description") or "",
            })
        else:
            entity_rows_think.append({"doc_id": doc_id, "name": str(e), "type": "", "description": ""})

    for trip in rels:
        if isinstance(trip, (list, tuple)) and len(trip) == 3:
            s, r, o = trip
            rel_rows_think.append({"doc_id": doc_id, "subject": str(s), "relation": str(r), "object": str(o)})
        elif isinstance(trip, dict):
            rel_rows_think.append({"doc_id": doc_id,
                                   "subject": str(trip.get("subject", "")),
                                   "relation": str(trip.get("relation", "")),
                                   "object": str(trip.get("object", ""))})
print("thinking pass:", len(entity_rows_think), "entities,", len(rel_rows_think), "relations")


INFO 08-18 17:23:50 [logger.py:41] Received request chatcmpl-dec9444d189d4159865f22ef81c2c0dc: prompt: '<|im_start|>system\nYour input fields are:\n1. `source_text` (str):\nYour output fields are:\n1. `entities` (list[str]): THOROUGH list of key entities\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## source_text ## ]]\n{source_text}\n\n[[ ## entities ## ]]\n{entities}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Extract key entities from the source text. Extracted entities are subjects or objects.\n        This is for an extraction task, please be THOROUGH and accurate to the reference text.<|im_end|>\n<|im_start|>user\n[[ ## source_text ## ]]\n/thinkRaven Space Systems Chooses Colorado for New Headquarters, Manufacturing Facility\n\nBROOMFIELD - Raven Space Systems, a 3D 

17:24:48 - LiteLLM:ERROR: litellm_logging.py:4483 - Error creating standard logging object - No module named 'fastapi_sso'
Traceback (most recent call last):
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 736, in completion
    raise e
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 664, in completion
    ) = self.make_sync_openai_chat_completion_request(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/logging_utils.py", line 149, in sync_wrapper
    result = func(*args, **kwargs)
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 482, in make_sync_openai_chat_completion_request
    raise e
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/llms/openai/openai.py", line 464, in make_sync_openai_chat_completion_request
    raw

INFO:     127.0.0.1:36080 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 08-18 17:24:48 [logger.py:41] Received request chatcmpl-49c530876c0b4a449a20ce3b8dfb5da3: prompt: '<|im_start|>system\nYour input fields are:\n1. `items` (set[str]): \n2. `context` (str): the larger context in which the items appear\nYour output fields are:\n1. `cluster` (set[str]):\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## items ## ]]\n{items}\n\n[[ ## context ## ]]\n{context}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "cluster": "{cluster}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"string\\"}, \\"uniqueItems\\": true}"\n}\nIn adhering to this structure, your objective is: \n        Find one cluster of related items from the list.\n        A cluster should contain items that are the same in meanin

2025/08/18 17:24:51 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


INFO 08-18 17:24:51 [logger.py:41] Received request chatcmpl-658f4bf37ea840e195337015691f8ba6: prompt: '<|im_start|>system\nYour input fields are:\n1. `items` (set[str]): \n2. `context` (str): the larger context in which the items appear\nYour output fields are:\n1. `cluster` (set[str]):\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## items ## ]]\n{items}\n\n[[ ## context ## ]]\n{context}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "cluster": "{cluster}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"string\\"}, \\"uniqueItems\\": true}"\n}\nIn adhering to this structure, your objective is: \n        Find one cluster of related items from the list.\n        A cluster should contain items that are the same in meaning, with different tenses, plural forms, stem forms, or cases. \n        

2025/08/18 17:25:28 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


INFO:     127.0.0.1:36080 - "POST /v1/chat/completions HTTP/1.1" 200 OK


ValidationError: 1 validation error for set[str]
  Input should be a valid set [type=set_type, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/set_type

INFO 08-18 17:25:31 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 67.9 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 6.9%
INFO 08-18 17:25:41 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 6.9%


In [ ]:
import pyarrow as pa
from pathlib import Path

def to_arrow_table(rows, schema):
    arrays = []
    names = [f.name for f in schema]
    cols = {n: [] for n in names}
    for r in rows:
        for n in names:
            cols[n].append(r.get(n))
    for n in names:
        arrays.append(pa.array(cols[n]))
    return pa.Table.from_arrays(arrays, names=names)

entity_schema = pa.schema([
    ("doc_id", pa.string()),
    ("name", pa.string()),
    ("type", pa.string()),
    ("description", pa.string()),
])
rel_schema = pa.schema([
    ("doc_id", pa.string()),
    ("subject", pa.string()),
    ("relation", pa.string()),
    ("object", pa.string()),
])

entities_tbl       = to_arrow_table(entity_rows, entity_schema)
relations_tbl      = to_arrow_table(rel_rows, rel_schema)
entities_tbl_think = to_arrow_table(entity_rows_think, entity_schema) if 'entity_rows_think' in globals() else None
relations_tbl_think= to_arrow_table(rel_rows_think, rel_schema) if 'rel_rows_think' in globals() else None

out_dir = Path("./data/processed"); out_dir.mkdir(parents=True, exist_ok=True)
ents_path = str(out_dir / "entities.parquet")
rels_path = str(out_dir / "relations.parquet")

con.register("entities_tbl", entities_tbl)
con.execute(f"COPY entities_tbl TO '{ents_path}' (FORMAT PARQUET)")

con.register("relations_tbl", relations_tbl)
con.execute(f"COPY relations_tbl TO '{rels_path}' (FORMAT PARQUET)")

if entities_tbl_think is not None:
    ents_think_path = str(out_dir / "entities_thinking.parquet")
    rels_think_path = str(out_dir / "relations_thinking.parquet")
    con.register("entities_tbl_think", entities_tbl_think)
    con.execute(f"COPY entities_tbl_think TO '{ents_think_path}' (FORMAT PARQUET)")
    con.register("relations_tbl_think", relations_tbl_think)
    con.execute(f"COPY relations_tbl_think TO '{rels_think_path}' (FORMAT PARQUET)")
    print("Wrote:\n ", ents_path, "\n ", rels_path, "\n ", ents_think_path, "\n ", rels_think_path)
else:
    print("Wrote:\n ", ents_path, "\n ", rels_path)


In [137]:
# Imports and configuration
import os
import json
import dspy
from kg_gen import KGGen
import pyarrow as pa
import pyarrow.parquet as pq
from openai import OpenAI

# Configuration
BASE_URL = "http://127.0.0.1:8000/v1"
SERVED_MODEL_NAME = "qwen3-1_7b-gptq"  # adjust to your model
CHUNK_SIZE = 2500  # characters per chunk

print(f"Configuration loaded: {BASE_URL}, Model: {SERVED_MODEL_NAME}")

Configuration loaded: http://127.0.0.1:8000/v1, Model: qwen3-1_7b-gptq


In [118]:
# Test basic vLLM connection without JSON constraints
def test_basic_vllm():
    client = OpenAI(base_url=BASE_URL, api_key="local-anything")
    
    try:
        response = client.chat.completions.create(
            model=SERVED_MODEL_NAME,
            messages=[
                {"role": "user", "content": "Say hello"}
            ],
            temperature=0.1,
            max_tokens=50,
            # NO response_format constraint
        )
        print("Basic test successful!")
        print(f"Response: {response.choices[0].message.content}")
        return True
    except Exception as e:
        print(f"Basic test failed: {e}")
        return False

# Run basic test
test_basic_vllm()

INFO 08-19 12:27:27 [logger.py:41] Received request chatcmpl-47465522c6db4f0e84947799fcb08f5d: prompt: '<|im_start|>user\nSay hello<|im_end|>\n<|im_start|>assistant\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.1, top_p=0.95, top_k=20, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=50, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None), prompt_token_ids: None, prompt_embeds shape: None, lora_request: None.
INFO 08-19 12:27:27 [async_llm.py:269] Added request chatcmpl-47465522c6db4f0e84947799fcb08f5d.
INFO:     127.0.0.1:36738 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Basic test successful!
Response: <think>
Okay, the user said "Say hello." I need to respond appropriately. Since I'm an AI assistant, I should 

True

In [138]:
# Method 1: Bypass DSPy's structured output (Most Reliable)
def setup_kggen_with_bypass():
    """
    Setup kg-gen with DSPy configured to bypass structured output issues
    """
    # Configure DSPy to use simple JSON mode without schemas
    lm = dspy.LM(
        f"openai/{SERVED_MODEL_NAME}",
        api_base=BASE_URL,
        api_key="local-anything",
        max_tokens=2048,  # Increase for better completions
        temperature=0.1,
        # Force JSON mode without complex schemas
        response_format={"type": "json_object"},
    )
    
    # Override DSPy settings
    dspy.settings.configure(
        lm=lm,
        experimental=True,  # Enable experimental features
    )
    
    # Initialize KGGen
    kg = KGGen(
        model=f"openai/{SERVED_MODEL_NAME}",
        temperature=0.1,
        api_key="EMPTY",
    )
    
    # Set environment variables for LiteLLM
    os.environ["OPENAI_API_BASE"] = BASE_URL
    os.environ["OPENAI_BASE_URL"] = BASE_URL
    os.environ["LITELLM_BASE_URL"] = BASE_URL
    os.environ["OPENAI_API_KEY"] = "EMPTY"
    
    return kg, lm

# Test setup
kg, lm = setup_kggen_with_bypass()
print("KGGen setup complete")

KGGen setup complete


In [139]:
# Method 2: Direct vLLM API without kg-gen (Fallback)
def extract_kg_direct_vllm(text, doc_id):
    """
    Direct KG extraction using vLLM without kg-gen
    """
    client = OpenAI(base_url=BASE_URL, api_key="local-anything")
    
    # Structured prompt for KG extraction
    prompt = f"""Extract entities and relationships from the following text.
Return ONLY a valid JSON object with this structure:
{{
  "entities": ["entity1", "entity2", ...],
  "relations": [
    ["subject", "predicate", "object"],
    ...
  ]
}}

Text: {text[:CHUNK_SIZE]}

JSON Output:"""

    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a knowledge graph extraction system. Return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=2048,
        response_format={"type": "json_object"}
    )
    
    try:
        result = json.loads(response.choices[0].message.content)
        return result.get("entities", []), result.get("relations", [])
    except Exception as e:
        print(f"JSON parse error: {e}")
        return [], []

# Test with sample text
test_text = "Apple Inc. CEO Tim Cook announced new products in California."
entities, relations = extract_kg_direct_vllm(test_text, "test_001")
print(f"Entities: {entities}")
print(f"Relations: {relations}")

INFO 08-19 13:08:15 [logger.py:41] Received request chatcmpl-b10dc9dcada14fdfa1b515dc1fff53de: prompt: '<|im_start|>system\nYou are a knowledge graph extraction system. Return only valid JSON.<|im_end|>\n<|im_start|>user\nExtract entities and relationships from the following text.\nReturn ONLY a valid JSON object with this structure:\n{\n  "entities": ["entity1", "entity2", ...],\n  "relations": [\n    ["subject", "predicate", "object"],\n    ...\n  ]\n}\n\nText: Apple Inc. CEO Tim Cook announced new products in California.\n\nJSON Output:<|im_end|>\n<|im_start|>assistant\n', params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.1, top_p=0.95, top_k=20, min_p=0.0, seed=None, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=2048, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided

In [ ]:
# Method 3: Monkey-patch DSPy to avoid problematic schemas
def patch_dspy_for_vllm():
    """
    Patch DSPy to avoid using uniqueItems in schemas
    """
    try:
        import dspy.adapters.json_adapter as json_adapter
        
        # Store original function
        original_get_schema = getattr(json_adapter, 'get_json_schema', None)
        
        def patched_get_schema(model, *args, **kwargs):
            schema = original_get_schema(model, *args, **kwargs) if original_get_schema else {}
            
            # Remove problematic keys recursively
            def clean_schema(obj):
                if isinstance(obj, dict):
                    # Remove uniqueItems key
                    obj.pop('uniqueItems', None)
                    # Recursively clean nested objects
                    for key, value in obj.items():
                        if isinstance(value, dict):
                            clean_schema(value)
                        elif isinstance(value, list):
                            for item in value:
                                if isinstance(item, dict):
                                    clean_schema(item)
                return obj
            
            return clean_schema(schema)
        
        # Apply patch
        if original_get_schema:
            json_adapter.get_json_schema = patched_get_schema
            print("DSPy patched successfully")
        else:
            print("Could not find get_json_schema to patch")
    except ImportError as e:
        print(f"Could not import json_adapter: {e}")

# Apply the patch
patch_dspy_for_vllm()

Could not find get_json_schema to patch


INFO 08-19 13:08:22 [loggers.py:122] Engine 000: Avg prompt throughput: 9.7 tokens/s, Avg generation throughput: 4.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [141]:
# Load your parquet file and inspect it
tbl = pq.read_table("test.parquet")
print(f"Table shape: {tbl.shape}")
print(f"Columns: {tbl.column_names}")

# Show first row
first_row = tbl.to_pylist()[0] if len(tbl) > 0 else None
if first_row:
    print("\nFirst document preview:")
    print(f"Doc ID: {first_row.get('doc_id', 'N/A')}")
    print(f"Title: {first_row.get('title', 'N/A')[:100]}...")
    print(f"Content length: {len(first_row.get('content', ''))}")

Table shape: (100, 6)
Columns: ['doc_id', 'title', 'content', 'source', 'published_at', 'url']

First document preview:
Doc ID: 8801998212
Title: Raven Space Systems Chooses Colorado for New Headquarters, Manufacturing Facility...
Content length: 1165


INFO 08-19 13:08:32 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [103]:
# Test processing a single document
def process_single_document(row):
    """Test function for a single document"""
    doc_id = row["doc_id"]
    text = f"{row.get('title', '')} {row.get('content', '')}"
    
    entity_rows = []
    rel_rows = []
    
    print(f"Processing doc {doc_id}...")
    
    # Try Method 1: kg-gen without clustering
    try:
        graph = kg.generate(
            input_data=text,
            context=("news article; "
                    "ONLY return JSON outputs; "
                    "cap to at most 80 relations"),  # keeps outputs bounded
            chunk_size=CHUNK,
            cluster=False,
            temperature=0.1,
        )
        
        if graph:
            # Extract entities
            entities = getattr(graph, "entities", graph.get("entities", []) if isinstance(graph, dict) else [])
            relations = getattr(graph, "relations", graph.get("relations", []) if isinstance(graph, dict) else [])
            
            print(f"  kg-gen extracted: {len(entities)} entities, {len(relations)} relations")
            return entities, relations
    except Exception as e:
        print(f"  kg-gen failed: {e}")
    
    # Fallback to Direct vLLM
    try:
        entities, relations = extract_kg_direct_vllm(text, doc_id)
        print(f"  Direct extraction: {len(entities)} entities, {len(relations)} relations")
        return entities, relations
    except Exception as e:
        print(f"  Direct extraction also failed: {e}")
        return [], []

# Test with first document
if first_row:
    test_entities, test_relations = process_single_document(first_row)
    print(f"\nSample entities: {test_entities if test_entities else 'None'}")
    print(f"Sample relations: {test_relations if test_relations else 'None'}")

Processing doc 8801998212...
  kg-gen extracted: 7 entities, 5 relations

Sample entities: {'Raven Space Systems', 'Microwave Assisted Deposition (MAD)', '392 jobs', 'Blake Herren', 'Broomfield, Colorado', '2,000 aerospace companies', 'aerospace ecosystem'}
Sample relations: {('Raven Space Systems', 'uses', 'Microwave Assisted Deposition (MAD)'), ('Raven Space Systems', 'CEO', 'Blake Herren'), ('Raven Space Systems', 'creates', '392 jobs'), ('aerospace ecosystem', 'home to', '2,000 aerospace companies'), ('Raven Space Systems', 'chooses', 'Broomfield, Colorado')}


In [ ]:
# Main processing function
def process_documents(parquet_file, limit=None):
    """
    Process documents from parquet file
    """
    # Read parquet
    tbl = pq.read_table(parquet_file)
    rows = tbl.to_pylist()
    
    if limit:
        rows = rows[:limit]
    
    entity_rows = []
    rel_rows = []
    
    for i, row in enumerate(rows):
        doc_id = row["doc_id"]
        text = f"{row.get('title', '')} {row.get('content', '')}"
        
        print(f"[{i+1}/{len(rows)}] Processing doc {doc_id}...")
        
        # Try extraction
        entities, relations = process_single_document(row)
        
        # Format entities
        for e in entities:
            if isinstance(e, dict):
                entity_rows.append({
                    "doc_id": doc_id,
                    "name": e.get("name", str(e)),
                    "type": e.get("type", ""),
                    "description": e.get("description", ""),
                })
            else:
                entity_rows.append({
                    "doc_id": doc_id,
                    "name": str(e),
                    "type": "",
                    "description": ""
                })
        
        # Format relations
        for rel in relations:
            if isinstance(rel, (list, tuple)) and len(rel) == 3:
                rel_rows.append({
                    "doc_id": doc_id,
                    "subject": str(rel[0]),
                    "relation": str(rel[1]),
                    "object": str(rel[2])
                })
    
    return entity_rows, rel_rows

# Test with first 10 documents
entity_rows, rel_rows = process_documents("test.parquet", limit=10)
print(f"\nExtracted {len(entity_rows)} entities and {len(rel_rows)} relations from 10 documents")

[1/10] Processing doc 8801998212...
Processing doc 8801998212...
INFO 08-19 13:08:44 [logger.py:41] Received request chatcmpl-95e274c0a3554d05ae8762f3bfb44d63: prompt: '<|im_start|>system\nYour input fields are:\n1. `source_text` (str):\nYour output fields are:\n1. `entities` (list[str]): THOROUGH list of key entities\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## source_text ## ]]\n{source_text}\n\n[[ ## entities ## ]]\n{entities}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Extract key entities from the source text. Extracted entities are subjects or objects.\n        This is for an extraction task, please be THOROUGH and accurate to the reference text.<|im_end|>\n<|im_start|>user\n[[ ## source_text ## ]]\nRaven Space Systems Chooses Colorado for New Headquarters, Manufa

2025/08/19 13:10:13 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
2025/08/19 13:10:13 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


INFO:     127.0.0.1:46272 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 08-19 13:10:13 [logger.py:41] Received request chatcmpl-c4776345e26a49cdbd0c32b695d042d0: prompt: '<|im_start|>system\nYour input fields are:\n1. `source_text` (str): \n2. `entities` (list[str]):\nYour output fields are:\n1. `relations` (list[tuple[str, str, str]]): List of subject-predicate-object tuples where subject and object are exact matches to items in entities list. BE THOROUGH\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## source_text ## ]]\n{source_text}\n\n[[ ## entities ## ]]\n{entities}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "relations": "{relations}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"array\\", \\"maxItems\\": 3, \\"minItems\\": 3, \\"prefixItems\\": [{\\"type\\": \\"string\\"},

2025/08/19 13:10:50 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.


INFO:     127.0.0.1:46272 - "POST /v1/chat/completions HTTP/1.1" 200 OK
  kg-gen extracted: 16 entities, 28 relations
[3/10] Processing doc 8802361313...
Processing doc 8802361313...
INFO 08-19 13:10:50 [logger.py:41] Received request chatcmpl-c30e1b6ab20f4a42adcfc8ea81e4aa4b: prompt: '<|im_start|>system\nYour input fields are:\n1. `source_text` (str):\nYour output fields are:\n1. `entities` (list[str]): THOROUGH list of key entities\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## source_text ## ]]\n{source_text}\n\n[[ ## entities ## ]]\n{entities}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Extract key entities from the source text. Extracted entities are subjects or objects.\n        This is for an extraction task, please be THOROUGH and accurate to the reference text.<|

...

INFO:     127.0.0.1:52430 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 08-19 13:17:28 [logger.py:41] Received request chatcmpl-15fea32170aa4cf7892af92b97a672db: prompt: '<|im_start|>system\nYour input fields are:\n1. `source_text` (str): \n2. `entities` (list[str]):\nYour output fields are:\n1. `relations` (list[tuple[str, str, str]]): List of subject-predicate-object tuples where subject and object are exact matches to items in entities list. BE THOROUGH\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\nInputs will have the following structure:\n\n[[ ## source_text ## ]]\n{source_text}\n\n[[ ## entities ## ]]\n{entities}\n\nOutputs will be a JSON object with the following fields.\n\n{\n  "relations": "{relations}        # note: the value you produce must adhere to the JSON schema: {\\"type\\": \\"array\\", \\"items\\": {\\"type\\": \\"array\\", \\"maxItems\\": 3, \\"minItems\\": 3, \\"prefixItems\\": [{\\"type\\": \\"string\\"},

13:17:32 - LiteLLM:ERROR: litellm_logging.py:4483 - Error creating standard logging object - No module named 'fastapi_sso'
Traceback (most recent call last):
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 4370, in get_standard_logging_object_payload
    clean_metadata = StandardLoggingPayloadSetup.get_standard_logging_metadata(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 3921, in get_standard_logging_metadata
    cold_storage_object_key = StandardLoggingPayloadSetup._generate_cold_storage_object_key(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 4109, in _generate_cold_storage_object_key
    configured_cold_storage_logger = ColdStorageHandler._get_configured_cold_storage_custom_logger()
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site

INFO 08-19 13:17:32 [loggers.py:122] Engine 000: Avg prompt throughput: 82.4 tokens/s, Avg generation throughput: 110.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.4%, Prefix cache hit rate: 36.6%
INFO:     127.0.0.1:52430 - "POST /v1/chat/completions HTTP/1.1" 200 OK
  kg-gen extracted: 17 entities, 13 relations

Extracted 94 entities and 100 relations from 10 documents


13:17:32 - LiteLLM:ERROR: litellm_logging.py:4483 - Error creating standard logging object - No module named 'fastapi_sso'
Traceback (most recent call last):
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 4370, in get_standard_logging_object_payload
    clean_metadata = StandardLoggingPayloadSetup.get_standard_logging_metadata(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 3921, in get_standard_logging_metadata
    cold_storage_object_key = StandardLoggingPayloadSetup._generate_cold_storage_object_key(
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/litellm_logging.py", line 4109, in _generate_cold_storage_object_key
    configured_cold_storage_logger = ColdStorageHandler._get_configured_cold_storage_custom_logger()
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site

INFO 08-19 13:17:42 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 2.2 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 36.6%
INFO 08-19 13:17:52 [loggers.py:122] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 36.6%


In [128]:
import dspy
import os
import shutil
import kg_gen

# 1. Check for DSPy's cache
print("Looking for DSPy cache...")
possible_cache_locations = [
    ".dspy_cache",
    "dspy_cache", 
    "cachedir",
    os.path.expanduser("~/.dspy/cache"),
    os.path.expanduser("~/.cache/dspy"),
    ".cache",
    "__pycache__",
]

for cache_dir in possible_cache_locations:
    if os.path.exists(cache_dir):
        print(f"Found cache: {cache_dir}")
        print(f"  Contents: {os.listdir(cache_dir)[:5]}...")  # Show first 5 files

# 2. Check if DSPy has a cache attribute
if hasattr(dspy, 'cache'):
    print(f"DSPy cache object: {dspy.cache}")
    
# 3. Check your kg_gen object
if 'kg_gen' in globals():
    print(f"kg_gen type: {type(kg_gen)}")
    if hasattr(kg_gen, '_cache'):
        print("kg_gen has a _cache attribute")
    if hasattr(kg_gen, 'clear_cache'):
        print("kg_gen has a clear_cache method")

Looking for DSPy cache...
DSPy cache object: <dspy.clients.cache.Cache object at 0x7584bffe2da0>
kg_gen type: <class 'module'>


In [131]:
import dspy

# See what methods the cache has
print("Cache methods and attributes:")
for attr in dir(dspy.cache):
    if not attr.startswith('_'):
        print(f"  {attr}")

# Check for common cache patterns
if hasattr(dspy.cache, 'cache'):
    print("\nFound cache.cache - type:", type(dspy.cache.cache))
    if hasattr(dspy.cache.cache, 'clear'):
        dspy.cache.cache.clear()
        print("✓ Cleared dspy.cache.cache")

if hasattr(dspy.cache, '_cache'):
    print("\nFound cache._cache - type:", type(dspy.cache._cache))
    if hasattr(dspy.cache._cache, 'clear'):
        dspy.cache._cache.clear()
        print("✓ Cleared dspy.cache._cache")

if hasattr(dspy.cache, 'data'):
    print("\nFound cache.data - type:", type(dspy.cache.data))
    if hasattr(dspy.cache.data, 'clear'):
        dspy.cache.data.clear()
        print("✓ Cleared dspy.cache.data")

# Try to see the internal structure
print("\nAll cache attributes (including private):")
for attr in dir(dspy.cache):
    try:
        value = getattr(dspy.cache, attr)
        if not callable(value) and not attr.startswith('__'):
            print(f"  {attr}: {type(value)}")
    except:
        pass

Cache methods and attributes:
  cache_key
  disk_cache
  enable_disk_cache
  enable_memory_cache
  get
  load_memory_cache
  memory_cache
  put
  reset_memory_cache
  save_memory_cache

All cache attributes (including private):
  _lock: <class '_thread.RLock'>
  disk_cache: <class 'diskcache.fanout.FanoutCache'>
  enable_disk_cache: <class 'bool'>
  enable_memory_cache: <class 'bool'>
  memory_cache: <class 'cachetools.LRUCache'>


In [133]:
import dspy

# 1. Clear the memory cache
dspy.cache.reset_memory_cache()
print("✓ Memory cache reset")

# 2. Clear the disk cache
dspy.cache.disk_cache.clear()
print("✓ Disk cache cleared")

# 3. Verify they're empty
print(f"Memory cache size: {len(dspy.cache.memory_cache)}")
print(f"Disk cache size: {len(dspy.cache.disk_cache)}")

# 4. Optionally disable them for future runs
# dspy.cache.enable_memory_cache = False
dspy.cache.enable_disk_cache = True
print("✓ Caching disabled")

# NOW run your processing - should take 10 minutes again!
entity_rows, rel_rows = process_documents("test.parquet", limit=10)

✓ Memory cache reset
✓ Disk cache cleared
Memory cache size: 0
Disk cache size: 0
✓ Caching disabled
🔄 FORCE REPROCESS MODE - Ignoring all previous processing state


[2025-08-19 13:03:39] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.492934 seconds



📖 Reading parquet file: test.parquet
  Limited to 10 documents
[1/10] Processing doc 8801998212...
  Hash: 26e7a55a, Length: 1247 chars
Processing doc 8801998212...


[2025-08-19 13:03:39] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.889727 seconds
[2025-08-19 13:03:40] INFO _base_client.py:1061: Retrying request to /chat/completions in 1.562194 seconds
13:03:42 - LiteLLM:ERROR: litellm_logging.py:4483 - Error creating standard logging object - No module named 'fastapi_sso'
Traceback (most recent call last):
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/home/jroberts/kg_extract/kge/.venv/lib/python3.10/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    r

KeyboardInterrupt: 

In [126]:
import os
import json
import time
import hashlib
from pathlib import Path

# Main processing function with force reprocessing
def process_documents(parquet_file, limit=None, force_reprocess=True, state_file=".processing_state.json"):
    """
    Process documents from parquet file with force reprocessing capability
    
    Args:
        parquet_file: Path to parquet file
        limit: Optional limit on number of documents
        force_reprocess: If True, ignore any previous processing state
        state_file: File to track processing state (set to None to disable)
    """
    
    # Load or initialize processing state
    processed_docs = set()
    if state_file and os.path.exists(state_file) and not force_reprocess:
        try:
            with open(state_file, 'r') as f:
                state = json.load(f)
                processed_docs = set(state.get("processed_docs", []))
                print(f"Loaded {len(processed_docs)} previously processed docs")
        except Exception as e:
            print(f"Warning: Could not load state file: {e}")
    
    if force_reprocess:
        print("🔄 FORCE REPROCESS MODE - Ignoring all previous processing state")
        processed_docs.clear()
        
        # Clear any potential caches
        cache_dirs = [
            ".cache", "__pycache__", "dspy_cache", 
            ".dspy_cache", ".kg_cache"
        ]
        for cache_dir in cache_dirs:
            if os.path.exists(cache_dir):
                print(f"  Clearing cache: {cache_dir}")
                import shutil
                shutil.rmtree(cache_dir, ignore_errors=True)
        
        # Clear environment caches
        os.environ["TRANSFORMERS_NO_CACHE"] = "1"
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        
        # Small delay to ensure clean state
        time.sleep(1)
    
    # Read parquet
    print(f"\n📖 Reading parquet file: {parquet_file}")
    tbl = pq.read_table(parquet_file)
    rows = tbl.to_pylist()
    
    if limit:
        rows = rows[:limit]
        print(f"  Limited to {limit} documents")
    
    entity_rows = []
    rel_rows = []
    
    # Track processing stats
    stats = {
        "processed": 0,
        "skipped": 0,
        "failed": 0,
        "retried": 0
    }
    
    for i, row in enumerate(rows):
        doc_id = row["doc_id"]
        text = f"{row.get('title', '')} {row.get('content', '')}"
        
        # Create document hash for change detection
        doc_hash = hashlib.md5(text.encode()).hexdigest()[:8]
        doc_key = f"{doc_id}_{doc_hash}"
        
        # Check if already processed (unless force mode)
        if not force_reprocess and doc_key in processed_docs:
            print(f"[{i+1}/{len(rows)}] Skipping doc {doc_id} (already processed)")
            stats["skipped"] += 1
            continue
        
        print(f"[{i+1}/{len(rows)}] Processing doc {doc_id}...")
        print(f"  Hash: {doc_hash}, Length: {len(text)} chars")
        
        # Try extraction with retries
        max_retries = 3
        entities = []
        relations = []
        success = False
        
        for attempt in range(max_retries):
            try:
                # Add debug flag to see what's happening inside
                if attempt > 0:
                    print(f"  Retry attempt {attempt + 1}/{max_retries}")
                    stats["retried"] += 1
                    time.sleep(2)  # Brief pause before retry
                
                # Clear any function-level caches before processing
                if hasattr(process_single_document, '__cache__'):
                    process_single_document.__cache__.clear()
                
                # Process with explicit new context
                entities, relations = process_single_document(row)
                
                # Validate results
                if entities is None or relations is None:
                    raise ValueError("Extraction returned None")
                
                success = True
                stats["processed"] += 1
                print(f"  ✓ Extracted: {len(entities)} entities, {len(relations)} relations")
                break
                
            except Exception as e:
                print(f"  ✗ Attempt {attempt + 1} failed: {e}")
                if attempt == max_retries - 1:
                    print(f"  ✗ Failed after {max_retries} attempts, skipping document")
                    stats["failed"] += 1
                    # Continue with empty results rather than crashing
                    entities = []
                    relations = []
        
        # Format entities
        for e in entities:
            if isinstance(e, dict):
                entity_rows.append({
                    "doc_id": doc_id,
                    "name": e.get("name", str(e)),
                    "type": e.get("type", ""),
                    "description": e.get("description", ""),
                })
            else:
                entity_rows.append({
                    "doc_id": doc_id,
                    "name": str(e),
                    "type": "",
                    "description": ""
                })
        
        # Format relations
        for rel in relations:
            if isinstance(rel, (list, tuple)) and len(rel) == 3:
                rel_rows.append({
                    "doc_id": doc_id,
                    "subject": str(rel[0]),
                    "relation": str(rel[1]),
                    "object": str(rel[2])
                })
        
        # Mark as processed (if tracking state)
        if success:
            processed_docs.add(doc_key)
            
            # Save state periodically (every 5 docs)
            if state_file and (i + 1) % 5 == 0:
                try:
                    with open(state_file, 'w') as f:
                        json.dump({"processed_docs": list(processed_docs)}, f)
                except Exception as e:
                    print(f"Warning: Could not save state: {e}")
    
    # Final state save
    if state_file:
        try:
            with open(state_file, 'w') as f:
                json.dump({"processed_docs": list(processed_docs)}, f)
            print(f"\n💾 Saved processing state to {state_file}")
        except Exception as e:
            print(f"Warning: Could not save final state: {e}")
    
    # Print summary statistics
    print("\n📊 Processing Summary:")
    print(f"  ✓ Processed: {stats['processed']} documents")
    print(f"  ⏭ Skipped: {stats['skipped']} documents")
    print(f"  ✗ Failed: {stats['failed']} documents")
    print(f"  🔄 Retried: {stats['retried']} times")
    print(f"  📝 Total Entities: {len(entity_rows)}")
    print(f"  🔗 Total Relations: {len(rel_rows)}")
    
    return entity_rows, rel_rows

# Test with first 10 documents - FORCE REPROCESSING
print("=" * 60)
print("STARTING FORCED REPROCESSING")
print("=" * 60)

# Force reprocess by setting force_reprocess=True
entity_rows, rel_rows = process_documents(
    "test.parquet", 
    limit=10,
    force_reprocess=True,  # This forces reprocessing
    state_file=None  # Set to None to disable state tracking entirely
)

print(f"\nExtracted {len(entity_rows)} entities and {len(rel_rows)} relations from 10 documents")

STARTING FORCED REPROCESSING
🔄 FORCE REPROCESS MODE - Ignoring all previous processing state


2025/08/19 12:54:20 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
[2025-08-19 12:54:20] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.444581 seconds



📖 Reading parquet file: test.parquet
  Limited to 10 documents
[1/10] Processing doc 8801998212...
  Hash: 26e7a55a, Length: 1247 chars
Processing doc 8801998212...
  kg-gen extracted: 7 entities, 5 relations
  ✓ Extracted: 7 entities, 5 relations
[2/10] Processing doc 8802509153...
  Hash: fe0c28ef, Length: 3322 chars
Processing doc 8802509153...
  kg-gen extracted: 16 entities, 8 relations
  ✓ Extracted: 16 entities, 8 relations
[3/10] Processing doc 8802361313...
  Hash: b191db55, Length: 2462 chars
Processing doc 8802361313...
  kg-gen failed: 'NoneType' object is not iterable


[2025-08-19 12:54:21] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.925014 seconds
2025/08/19 12:54:22 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
[2025-08-19 12:54:22] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.410127 seconds


  Direct extraction also failed: Connection error.
  ✓ Extracted: 0 entities, 0 relations
[4/10] Processing doc 8802140145...
  Hash: 1ae2baf6, Length: 2607 chars
Processing doc 8802140145...
  kg-gen failed: 'NoneType' object is not iterable


...

  Direct extraction also failed: Connection error.
  ✓ Extracted: 0 entities, 0 relations
[9/10] Processing doc 8796482557...
  Hash: 3d4d23b6, Length: 3781 chars
Processing doc 8796482557...
  kg-gen failed: 'NoneType' object is not iterable


[2025-08-19 12:54:25] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.957955 seconds
2025/08/19 12:54:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=4000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.1)  if the reason for truncation is repetition.
[2025-08-19 12:54:26] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.434207 seconds


  Direct extraction also failed: Connection error.
  ✓ Extracted: 0 entities, 0 relations
[10/10] Processing doc 8803587492...
  Hash: eb5044e2, Length: 4056 chars
Processing doc 8803587492...
  kg-gen failed: 'NoneType' object is not iterable


[2025-08-19 12:54:26] INFO _base_client.py:1061: Retrying request to /chat/completions in 0.838576 seconds


  Direct extraction also failed: Connection error.
  ✓ Extracted: 0 entities, 0 relations

📊 Processing Summary:
  ✓ Processed: 10 documents
  ⏭ Skipped: 0 documents
  ✗ Failed: 0 documents
  🔄 Retried: 0 times
  📝 Total Entities: 43
  🔗 Total Relations: 23

Extracted 43 entities and 23 relations from 10 documents


In [143]:
from collections import defaultdict

# Assuming your list is called 'relations'
grouped_by_doc = defaultdict(list)

for rel in rel_rows:
    doc_id = rel['doc_id']
    # Create a copy without doc_id
    rel_without_id = {k: v for k, v in rel.items() if k != 'doc_id'}
    grouped_by_doc[doc_id].append(rel_without_id)

# Convert to regular dict
grouped_by_doc = dict(grouped_by_doc)


In [112]:
for doc_id, relations in grouped_by_doc.items():
    print(f"Document ID: {doc_id}")
    for rel in relations:
        print(f"  Relation: {rel}")
    print()  # Newline for better readability

Document ID: 8801998212
  Relation: {'subject': 'Raven Space Systems', 'relation': 'uses', 'object': 'Microwave Assisted Deposition (MAD)'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'CEO', 'object': 'Blake Herren'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'creates', 'object': '392 jobs'}
  Relation: {'subject': 'aerospace ecosystem', 'relation': 'home to', 'object': '2,000 aerospace companies'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'chooses', 'object': 'Broomfield, Colorado'}

Document ID: 8802509153
  Relation: {'subject': 'US Army', 'relation': 'releases', 'object': '2025 strategy'}
  Relation: {'subject': 'US Army', 'relation': 'releases', 'object': '2040 strategy'}
  Relation: {'subject': 'homeland missile defense', 'relation': 'growing involvement', 'object': 'Ground-based Midcourse Defense'}
  Relation: {'subject': 'artificial intelligence', 'relation': 'incorporating', 'object': 'operator overload'}
  Relation: {'subject': 

In [144]:
for doc_id, relations in grouped_by_doc.items():
    print(f"Document ID: {doc_id}")
    for rel in relations:
        print(f"  Relation: {rel}")
    print()  # Newline for better readability

Document ID: 8801998212
  Relation: {'subject': 'Raven Space Systems', 'relation': 'joins', 'object': 'aerospace ecosystem'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'chooses', 'object': 'Broomfield, Colorado'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'creates', 'object': '392 jobs'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'has', 'object': 'aerospace ecosystem'}
  Relation: {'subject': 'Blake Herren', 'relation': 'says', 'object': 'Raven Space Systems'}
  Relation: {'subject': '392 jobs', 'relation': 'created', 'object': 'Raven Space Systems'}
  Relation: {'subject': 'Raven Space Systems', 'relation': 'selects', 'object': 'Broomfield, Colorado'}
  Relation: {'subject': 'Microwave Assisted Deposition (MAD)', 'relation': 'technology', 'object': 'Raven Space Systems'}

Document ID: 8802509153
  Relation: {'subject': 'operator overload', 'relation': 'area', 'object': 'artificial intelligence'}
  Relation: {'subject': 'interceptor on in